In [ ]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 12.1 MB/s eta 0:00:00


In [9]:
from scipy.sparse import data
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
import pandas as pd
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score
import optuna

# Load dataset
data = load_breast_cancer()
X, y = data.data, data.target

# Split dataset into train and test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Standardize features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print(f"Training data shape: {X_train.shape}")
print(f"Test data shape: {X_test.shape}")

# Train a baseline XGBoost model
baseline_model = XGBClassifier(eval_metric='logloss', random_state=42)
baseline_model.fit(X_train, y_train)

# Evaluate the model
baseline_preds = baseline_model.predict(X_test)
beseline_accuracy = accuracy_score(y_test, baseline_preds)
print(f"Baseline XGBoost Accuracy: {beseline_accuracy:.4f}")

# Define the objective function for Optuna
def objective(trial):
  params = {
      'n_estimators': trial.suggest_int('n_estimators', 50, 500),
      'max_depth': trial.suggest_int('max_depth', 3, 100),
      'learning_rate': trial.suggest_int('learning_rate', 0.01, 0.3),
      'subsample': trial.suggest_int('subsample', 0.6, 1.0),
      'colsample_bytree': trial.suggest_int('colsample_bytree', 0.6, 1.0),
      'gamma': trial.suggest_int('gamma', 0, 10),
      'reg_alpha': trial.suggest_int('reg_alpha', 0, 10),
      'reg_lambda': trial.suggest_int('reg_lambda', 0, 10)
  }

  # Train XGBoost model with suggested params
  model = XGBClassifier(eval_metric='logloss', random_state=42, **params)
  model.fit(X_train, y_train)

  # Evaluate model on validation set
  preds = model.predict(X_test)
  accuracy = accuracy_score(y_test, preds)
  return accuracy

# Create an Optuna study
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50)

# Best hyperparameters
print("Best Hyperparameters:", study.best_params)
print("Optuna Best Accuracy: ", study.best_value)

# Define parameter grid
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.2],
    'subsample': [0.6, 0.8, 1.0]
}

# Train XGBoost with Grid Search
grid_search = GridSearchCV(
    estimator=XGBClassifier(eval_metric='logloss', random_state=42),
    param_grid=param_grid,
    scoring='accuracy',
    cv=5,
    verbose=1
)
grid_search.fit(X_train, y_train)

# Best parameters and accuracy
print("\n\n\nGrid Search Parameters: ", grid_search.best_params_)
print("Grid Search Best Accuracy:", grid_search.best_score_)

# Define parameter distributions
param_dist = {
    'n_estimators': [50, 100, 200, 300, 400],
    'max_depth': [3, 5, 7, 9],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'subsample': [0.6, 0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.6, 0.7, 0.8, 0.9, 1.0]
}

# Train XGBoost with Random Search
random_search = RandomizedSearchCV(
    estimator=XGBClassifier(eval_metric='logloss', random_state=42),
    param_distributions=param_dist,
    n_iter=50,
    scoring='accuracy',
    cv=3,
    verbose=1,
    random_state=42
)
random_search.fit(X_train, y_train)

# Best parameters and accuracy
print("\n\n\nRandom Search Best Parameters:", random_search.best_params_)
print("Random Search Best Accuracy:", random_search.best_score_)






[I 2026-08-01 16:10:21,182] A new study created in memory with name: no-name-6a4b5d3f-62dd-41ab-aa63-33d1db549bfb
[I 2026-08-01 16:10:21,212] Trial 0 finished with value: 0.6228070175438597 and parameters: {'n_estimators': 176, 'max_depth': 96, 'learning_rate': 0, 'subsample': 0, 'colsample_bytree': 0, 'gamma': 2, 'reg_alpha': 6, 'reg_lambda': 9}. Best is trial 0 with value: 0.6228070175438597.


Training data shape: (455, 30)
Test data shape: (114, 30)
Baseline XGBoost Accuracy: 0.9649


[I 2026-08-01 16:10:21,659] Trial 1 finished with value: 0.6228070175438597 and parameters: {'n_estimators': 492, 'max_depth': 16, 'learning_rate': 0, 'subsample': 1, 'colsample_bytree': 1, 'gamma': 5, 'reg_alpha': 7, 'reg_lambda': 10}. Best is trial 0 with value: 0.6228070175438597.
[I 2026-08-01 16:10:21,716] Trial 2 finished with value: 0.6228070175438597 and parameters: {'n_estimators': 455, 'max_depth': 14, 'learning_rate': 0, 'subsample': 0, 'colsample_bytree': 1, 'gamma': 3, 'reg_alpha': 2, 'reg_lambda': 5}. Best is trial 0 with value: 0.6228070175438597.
[I 2026-08-01 16:10:22,160] Trial 3 finished with value: 0.6228070175438597 and parameters: {'n_estimators': 468, 'max_depth': 42, 'learning_rate': 0, 'subsample': 1, 'colsample_bytree': 1, 'gamma': 6, 'reg_alpha': 0, 'reg_lambda': 5}. Best is trial 0 with value: 0.6228070175438597.
[I 2026-08-01 16:10:22,185] Trial 4 finished with value: 0.6228070175438597 and parameters: {'n_estimators': 148, 'max_depth': 36, 'learning_rate':

Best Hyperparameters: {'n_estimators': 176, 'max_depth': 96, 'learning_rate': 0, 'subsample': 0, 'colsample_bytree': 0, 'gamma': 2, 'reg_alpha': 6, 'reg_lambda': 9}
Optuna Best Accuracy:  0.6228070175438597
Fitting 5 folds for each of 81 candidates, totalling 405 fits



Grid Search Parameters:  {'learning_rate': 0.2, 'max_depth': 5, 'n_estimators': 100, 'subsample': 0.6}
Grid Search Best Accuracy: 0.9780219780219781
Fitting 3 folds for each of 50 candidates, totalling 150 fits



Random Search Best Parameters: {'subsample': 0.7, 'n_estimators': 100, 'max_depth': 9, 'learning_rate': 0.2, 'colsample_bytree': 1.0}
Random Search Best Accuracy: 0.9758045776693388
